In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Dimensionality Reduction of GEE Features File

In [ ]:
import pandas as pd

chunk_size = 10000  # Set the desired chunk size

# Missing Value Ratio
threshold_missing = 0.8  # Set threshold for missing value ratio

# Low Variance Filter
threshold_variance = 0.1  # Set threshold for variance

# High Correlation Filter
threshold_correlation = 0.9  # Set threshold for correlation

# List to store filtered chunks
filtered_chunks = []

# Iterate over the dataset in chunks
for chunk in pd.read_csv('/kaggle/input/maternal-and-child-health-monitoring-in-lmics/gee_features.csv', chunksize=chunk_size, low_memory = False):
    # Missing Value Ratio
    missing_ratio = chunk.isnull().sum() / len(chunk)
    missing_cols = missing_ratio[missing_ratio > threshold_missing].index
    chunk = chunk.drop(missing_cols, axis=1)

    # Low Variance Filter
    variances = chunk.var()
    low_variance_cols = variances[variances < threshold_variance].index
    chunk = chunk.drop(low_variance_cols, axis=1)

    # High Correlation Filter
    corr_matrix = chunk.corr().abs()
    upper_tri = corr_matrix.where(~np.tril(np.ones(corr_matrix.shape)).astype(np.bool))
    high_correlation_cols = [col for col in upper_tri.columns if any(upper_tri[col] > threshold_correlation)]
    chunk = chunk.drop(high_correlation_cols, axis=1)

    # Append the filtered chunk to the list
    filtered_chunks.append(chunk)

# Combine filtered chunks back together
combined_data = pd.concat(filtered_chunks, axis=0)

# Get the consistent feature set
common_features = set.intersection(*[set(chunk.columns) for chunk in filtered_chunks])
combined_data = combined_data[list(common_features)]

combined_data.head()

In [ ]:
combined_data.shape
#(120984, 901)

In [ ]:
combined_data.to_csv('gee_features_Dimenreduced.csv' , index = False)

## Splitting the reduced GEE Features file into Traning Data and Sample Data

In [ ]:
gee_reduced = pd.read_csv('/kaggle/input/dimenreduced/gee_features_Dimenreduced.csv', low_memory = False)
gee_reduced.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Iterate over each column in the DataFrame
for column in gee_reduced.columns:
    if column != 'DHSID' and gee_reduced[column].dtype == 'object':  # Check if the column is non-numeric and not 'DHSID'
        gee_reduced[column] = label_encoder.fit_transform(gee_reduced[column].astype(str))

# Print the updated DataFrame
gee_reduced.head()

In [ ]:
dhsid = gee_reduced['DHSID']
dhsid

In [ ]:
from sklearn.impute import SimpleImputer

# Create a copy of the DataFrame without the 'DHSID' column
gee_reduced_no_dhsid = gee_reduced.drop('DHSID', axis=1)

# Create a SimpleImputer object
imputer = SimpleImputer(strategy='mean')

# Impute missing values without including the 'DHSID' column
df_imputed_no_dhsid = pd.DataFrame(imputer.fit_transform(gee_reduced_no_dhsid), columns=gee_reduced_no_dhsid.columns)

# Combine the imputed data with the 'DHSID' column
df_imputed = pd.concat([dhsid, df_imputed_no_dhsid], axis=1)

# Print the imputed DataFrame
df_imputed.head()

In [ ]:
sample_sub = pd.read_csv('/kaggle/input/maternal-and-child-health-monitoring-in-lmics/sample submission.csv', low_memory = False)
training_label = pd.read_csv('/kaggle/input/maternal-and-child-health-monitoring-in-lmics/training_label.csv', low_memory = False)

In [ ]:
gee_reduced_training = pd.merge(df_imputed , training_label.drop(columns=['LONGNUM', 'URBAN_RURA', 'LATNUM']) ,  on ='DHSID', how='inner')
gee_reduced_sample = pd.merge(df_imputed, sample_sub, on = 'DHSID', how = 'inner')
gee_reduced_training.head()

In [ ]:
gee_reduced_training.shape
#(101140, 909)

In [ ]:
gee_reduced_training['DHSID'].value_counts()

In [ ]:
gee_reduced_sample.shape
#(15346, 907)

In [ ]:
gee_reduced_sample.drop_duplicates(subset = ['DHSID'] , inplace = True , keep = 'first')
gee_reduced_sample.shape

In [ ]:
missing_dhsids = list(set(sample_sub.DHSID.values) - set(gee_reduced_sample.DHSID.values))
missing_dhsids

'''['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']'''

In [ ]:
gee_reduced_training.drop_duplicates(inplace = True, keep = 'first')
gee_reduced_training.shape

In [ ]:
# Find extra columns in gee_reduced_training
extra_columns_training = list(set(gee_reduced_training.columns) - set(gee_reduced_sample.columns))

# Find extra columns in gee_reduced_sample
extra_columns_sample = list(set(gee_reduced_sample.columns) - set(gee_reduced_training.columns))

# Print the extra column names
print("Extra columns in gee_reduced_training:", extra_columns_training)
print("Extra columns in gee_reduced_sample:", extra_columns_sample)

#Extra columns in gee_reduced_training: ['DHSCLUST', 'DHSYEAR']
#Extra columns in gee_reduced_sample: []

In [ ]:
gee_reduced_training.drop(extra_columns_training, inplace = True , axis = 1)
print(gee_reduced_training.shape)
print(gee_reduced_sample.shape)
#(101138, 907)
#(14975, 907)

In [ ]:
gee_reduced_training.to_csv('gee_reduced_training.csv' , index = False)
gee_reduced_sample.to_csv('gee_reduced_sample.csv', index = False)

## Label-Wise Model Training

***Mean_BMI***

In [ ]:
!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
training

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates


In [ ]:
dhsids_training = training_without_duplicates['DHSID']
dhsids_training

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]

In [ ]:
labels = training_without_duplicates.iloc[:, -6:]
labels

In [ ]:
for col in labels.columns:
    print(f'Null in {col}: {labels[col].isna().sum()}')
    
'''Null in Mean_BMI: 18491
Null in Median_BMI: 18491
Null in Unmet_Need_Rate: 1863
Null in Under5_Mortality_Rate: 28840
Null in Skilled_Birth_Attendant_Rate: 32476
Null in Stunted_Rate: 57209'''

In [ ]:
import pandas as pd

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Identify rows with null values in the 'Median_BMI' column
null_rows = training_without_duplicates[training_without_duplicates['Mean_BMI'].isnull()]

# Set the random seed (random state) for reproducibility
random_state = 42

# Number of rows to randomly remove
rows_to_remove = 16000

# Get the indices of random null-valued rows to remove
rows_to_remove_indices = null_rows.sample(n=rows_to_remove, random_state=random_state).index

# Remove the randomly selected null-valued rows
final_selected_rows = training_without_duplicates.drop(rows_to_remove_indices)

# Calculate the mean of 'Median_BMI' for imputation
median_bmi_mean = final_selected_rows['Mean_BMI'].mean()

# Impute remaining null values in 'Median_BMI' with the mean value
final_selected_rows['Mean_BMI'].fillna(median_bmi_mean, inplace=True)

# The 'remaining_rows' DataFrame now contains the rows with non-null 'Median_BMI'
# after removing 16,000 random null-valued rows and imputing the remaining 2,000 null values with the mean.

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
sample

In [ ]:
dhsids_sample = sample['DHSID']
dhsids_sample

In [ ]:
final_selected_rows.drop('DHSID',axis = 1, inplace = True)
final_selected_rows

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X = final_selected_rows.iloc[:, :-6]
y = final_selected_rows.loc[:, ['Mean_BMI']]

# Define the number of folds for k-fold cross-validation
n_folds = 3

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
        "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
        "random_state": 42,
        "objective": "reg:squarederror",  # Regression task
        "eval_metric": "rmse",  # Root Mean Squared Error metric
        "tree_method": "gpu_hist"
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = np.zeros(len(X))
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
        oof_predictions[val_idx] = predictions

        rmse = mean_squared_error(y_val, predictions, squared=False)
        rmses.append(rmse)
        
        # Report intermediate result for pruning
        trial.report(rmse, step=len(rmses))
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()


    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = np.mean(rmses)

    return cv_rmse


study_xgb = optuna.create_study(direction='minimize')
pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
study_xgb.optimize(objective, n_trials=500)

# After hyperparameter tuning, you can train the final model on the entire training data.
# Use the best hyperparameters obtained from the Optuna study.

In [ ]:
'''best_params = {'n_estimators': 1021,
               'learning_rate': 0.007205299668408112,
               'max_depth': 10, 
               'subsample': 0.9883355319954155,
               'colsample_bytree': 0.3809596334310913,
               'min_child_weight': 12,
               'gamma': 0.0028943412540592683,
               'reg_alpha': 0.6123215976798917,
               'reg_lambda': 0.7185694163814978,
               "tree_method": "gpu_hist", 
               'random_state' : 42}'''

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Mean_BMI']

In [ ]:
# Get the best hyperparameters for the current label
best_params = study_xgb.best_params
# Train a new model using the best hyperparameters for the current label
final_xgboost_model = xgb.XGBRegressor(**best_params)
final_xgboost_model.fit(X, y)
# Make predictions for the current label
predictions = model.predict(X_sample)

In [ ]:
cols = ["Mean_BMI"]

predictions_cat = pd.DataFrame(predictions , columns = cols)
predictions_cat

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)
df

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1)

# Print the merged dataframe
pred_merge_df

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

# Print the merged DataFrame
merged_df

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())
merged_df

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)
merged_df

In [ ]:
merged_df.to_csv('XGB_(16k)kNull_RS42_MeanBMI.csv' , index = False)

***Median_BMI***

In [ ]:
pip install gdown

In [ ]:
!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
training

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates

In [ ]:
dhsids_training = training_without_duplicates['DHSID']
dhsids_training

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]

'''Index(['Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate', 'Under5_Mortality_Rate',
       'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
      dtype='object')'''

In [ ]:
labels = training_without_duplicates.iloc[:, -6:]
labels

In [ ]:
for col in labels.columns:
    print(f'Null in {col}: {labels[col].isna().sum()}')
    
'''Null in Mean_BMI: 18491
Null in Median_BMI: 18491
Null in Unmet_Need_Rate: 1863
Null in Under5_Mortality_Rate: 28840
Null in Skilled_Birth_Attendant_Rate: 32476
Null in Stunted_Rate: 57209'''

In [ ]:
import pandas as pd

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Identify rows with null values in the 'Median_BMI' column
null_rows = training_without_duplicates[training_without_duplicates['Median_BMI'].isnull()]

# Set the random seed (random state) for reproducibility
random_state = 42

# Number of rows to randomly remove
rows_to_remove = 16000

# Get the indices of random null-valued rows to remove
rows_to_remove_indices = null_rows.sample(n=rows_to_remove, random_state=random_state).index

# Remove the randomly selected null-valued rows
final_selected_rows = training_without_duplicates.drop(rows_to_remove_indices)

# Calculate the mean of 'Median_BMI' for imputation
median_bmi_mean = final_selected_rows['Median_BMI'].mean()

# Impute remaining null values in 'Median_BMI' with the mean value
final_selected_rows['Median_BMI'].fillna(median_bmi_mean, inplace=True)

final_selected_rows

# The 'remaining_rows' DataFrame now contains the rows with non-null 'Median_BMI'
# after removing 16,000 random null-valued rows and imputing the remaining 2,000 null values with the mean.

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
sample

In [ ]:
dhsids_sample = sample['DHSID']
dhsids_sample

In [ ]:
final_selected_rows.drop('DHSID',axis = 1, inplace = True)
final_selected_rows

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X = final_selected_rows.iloc[:, :-6]
y = final_selected_rows.loc[:, ['Median_BMI']]

# Define the number of folds for k-fold cross-validation
n_folds = 3

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
        "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
        "random_state": 42,
        "objective": "reg:squarederror",  # Regression task
        "eval_metric": "rmse",  # Root Mean Squared Error metric
        "tree_method": "gpu_hist"
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = np.zeros(len(X))
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
        oof_predictions[val_idx] = predictions

        rmse = mean_squared_error(y_val, predictions, squared=False)
        rmses.append(rmse)
        
        # Report intermediate result for pruning
        trial.report(rmse, step=len(rmses))
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()


    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = np.mean(rmses)

    return cv_rmse


study_xgb = optuna.create_study(direction='minimize')
pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
study_xgb.optimize(objective, n_trials=500)

# After hyperparameter tuning, you can train the final model on the entire training data.
# Use the best hyperparameters obtained from the Optuna study.

In [ ]:
#Trial 408 finished with value: 2.1249550224769043
'''best_params: {'n_estimators': 908, 
              'learning_rate': 0.008572596514064812, 
              'max_depth': 10,
              'subsample': 0.9806138553667393,
              'colsample_bytree': 0.3555037506225436,
              'min_child_weight': 17, 
              'gamma': 0.8512650980611904, 
              'reg_alpha': 0.5173019013846165, 
              'reg_lambda': 0.4812331516964748}'''
#Best is trial 408 with value: 2.1249550224769043.

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Median_BMI']

In [ ]:
# Get the best hyperparameters for the current label
best_params = study_xgb.best_params
# Train a new model using the best hyperparameters for the current label
final_xgboost_model = xgb.XGBRegressor(**best_params)
final_xgboost_model.fit(X, y)
# Make predictions for the current label
predictions = final_xgboost_model.predict(X_sample)

In [ ]:
cols = ["Median_BMI"]

predictions_cat = pd.DataFrame(predictions , columns = cols)
predictions_cat

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)
df

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1)

# Print the merged dataframe
pred_merge_df

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

# Print the merged DataFrame
merged_df

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())
merged_df

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)
merged_df

In [ ]:
merged_df.to_csv('XGB_(18-2)kNull_RS42_MedianBMI.csv' , index = False)

***Unmet_Need_Rate***

In [ ]:
!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
training

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates

In [ ]:
dhsids_training = training_without_duplicates['DHSID']
dhsids_training

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]

'''Index(['Mean_BMI', 'Median_BMI', 'Unmet_Need_Rate', 'Under5_Mortality_Rate',
       'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
      dtype='object')'''

In [ ]:
labels = training_without_duplicates.iloc[:, -6:]
labels

In [ ]:
for col in labels.columns:
    print(f'{col}: {labels[col].isna().sum()}')
    
    
'''Mean_BMI: 18491
Median_BMI: 18491
Unmet_Need_Rate: 1863
Under5_Mortality_Rate: 28840
Skilled_Birth_Attendant_Rate: 32476
Stunted_Rate: 57209'''

In [ ]:
import numpy as np
import pandas as pd

# Assuming training_without_duplicates is your DataFrame
# Remove the existing 1800 rows with null values
training_without_duplicates.dropna(subset=['Unmet_Need_Rate'], inplace=True)

# Remove an additional 2000 random rows using random state 42
selected_rows = training_without_duplicates.sample(n=2000, random_state=42)
training_without_duplicates.drop(selected_rows.index, inplace=True)

# Impute any remaining null values with the mean
mean_bmi = training_without_duplicates['Unmet_Need_Rate'].mean()
training_without_duplicates['Unmet_Need_Rate'].fillna(mean_bmi, inplace=True)

In [ ]:
training_without_duplicates.shape
#(94348, 907)

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]
'''Index(['Mean_BMI', 'Median_BMI', 'Under5_Mortality_Rate',
       'Skilled_Birth_Attendant_Rate', 'Stunted_Rate'],
      dtype='object')'''

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
sample

In [ ]:
dhsids_sample = sample['DHSID']
dhsids_sample

In [ ]:
training_without_duplicates.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
#import catboost as cb
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb


# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X = training_without_duplicates.iloc[:, :-6]
y = training_without_duplicates.loc[:, ['Unmet_Need_Rate']]

# Define the number of folds for k-fold cross-validation
n_folds = 3

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),  # Expanded range
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),  # Expanded range
        "max_depth": trial.suggest_int("max_depth", 3, 20),  # Expanded range
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Expanded range
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),  # Expanded range
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),  # Expanded range
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),  # Expanded range
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),  # Expanded range
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),  # Expanded range
        "random_state": 42,
        "objective": "reg:squarederror",  # Regression task
        "eval_metric": "rmse",  # Root Mean Squared Error metric
        "tree_method": "gpu_hist"
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = np.zeros(len(X))
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
        oof_predictions[val_idx] = predictions

        rmse = mean_squared_error(y_val, predictions, squared=False)
        rmses.append(rmse)
        
        # Report intermediate result for pruning
        trial.report(rmse, step=len(rmses))
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()


    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = np.mean(rmses)

    return cv_rmse


study_xgb = optuna.create_study(direction='minimize')
pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
study_xgb.optimize(objective, n_trials=500)

# After hyperparameter tuning, you can train the final model on the entire training data.
# Use the best hyperparameters obtained from the Optuna study

In [ ]:
#Trial 123 finished with value: 19.07345502434539 
'''best_params = {'n_estimators': 1494, 
               'learning_rate': 0.006122420799116568,
               'max_depth': 11, 
               'subsample': 0.9709483301720011, 
               'colsample_bytree': 0.5439318691481286, 
               'min_child_weight': 19,
               'gamma': 0.5836594670780896,
               'reg_alpha': 0.6926816720550927, 
               'reg_lambda': 0.057458000729569705,
               "tree_method": "gpu_hist",
               "random_state": 42
              }'''
#Best is trial 123 with value: 19.07345502434539.\n"

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Unmet_Need_Rate']

In [ ]:
# Get the best hyperparameters for the current label
best_params = study_xgb.best_params
# Train a new model using the best hyperparameters for the current label
final_xgboost_model = xgb.XGBRegressor(**best_params)
final_xgboost_model.fit(X, y)

In [ ]:
# Make predictions for the current label
predictions = final_xgboost_model.predict(X_sample)
predictions

In [ ]:
cols = ["Unmet_Need_Rate"]

predictions_cat = pd.DataFrame(predictions , columns = cols)
predictions_cat

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)
df

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1)

# Print the merged dataframe
pred_merge_df

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

# Print the merged DataFrame
merged_df

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())
merged_df

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)
merged_df

In [ ]:
merged_df.to_csv('XGB_Unmet_tuned_3kRowsRemoved_RS42.csv' , index = False)

***Under5_Mortality_Rate***

In [ ]:
training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)



In [ ]:
training_without_duplicates.isnull().sum()

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
import numpy as np

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Randomly remove 22,000 rows with null values, using random state 42 for reproducibility
null_rows = training_without_duplicates[training_without_duplicates['Under5_Mortality_Rate'].isnull()]
random_null_rows = null_rows.sample(n=5000, random_state=42)
training_without_duplicates = training_without_duplicates.drop(random_null_rows.index)

# Impute remaining null values using mean imputation
imputer = SimpleImputer(strategy='mean')
imputed_values = imputer.fit_transform(training_without_duplicates[['Under5_Mortality_Rate']])
training_without_duplicates['Under5_Mortality_Rate'] = imputed_values

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)

In [ ]:
dhsids_sample = sample['DHSID']

In [ ]:
training_without_duplicates.drop('DHSID',axis = 1, inplace = True)

In [ ]:
selected_rows = training_without_duplicates.copy()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb

# Assuming 'training_without_duplicates' is your DataFrame after preprocessing

# Splitting the data into features (X) and labels (y)
X = training_without_duplicates.iloc[:, :-6]  # Exclude the 'Skilled_Birth_Attendant_Rate' column
y = training_without_duplicates['Under5_Mortality_Rate']

# Define the number of folds for k-fold cross-validation
n_folds = 3

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
        "random_state": 42,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "gpu_hist"
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = np.zeros(len(X))
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques
        oof_predictions[val_idx] = predictions

        rmse = mean_squared_error(y_val, predictions, squared=False)
        rmses.append(rmse)
        
        # Report intermediate result for pruning
        trial.report(rmse, step=len(rmses))
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = np.mean(rmses)

    return cv_rmse

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective, n_trials=400)

# After hyperparameter tuning, you can train the final model on the entire training data.
# Use the best hyperparameters obtained from the Optuna study.


In [ ]:
#Trial 82 finished with value: 4.766250927332784 
'''best_params = {'n_estimators': 1279,
               'learning_rate': 0.008244534414963565,
               'max_depth': 10, 
               'subsample': 0.9977006392316401, 
               'colsample_bytree': 0.34089086689526177, 
               'min_child_weight': 19, 
               'gamma': 0.39657616554247066,
               'reg_alpha': 0.5993129929719778, 
               'reg_lambda': 0.6119614886920933}'''
#Best is trial 82 with value: 4.766250927332784.

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Under5_Mortality_Rate']

In [ ]:
# Get the best hyperparameters for the current label
best_params = study_xgb.best_params
# Train a new model using the best hyperparameters for the current label
final_xgboost_model = xgb.XGBRegressor(**best_params)
final_xgboost_model.fit(X, y)
# Make predictions for the current label
predictions = final_xgboost_model.predict(X_sample)

In [ ]:
cols = ["Under5_Mortality_Rate"]

predictions_cat = pd.DataFrame(predictions , columns = cols)

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)

In [ ]:
merged_df.to_csv('XGB_5K_null_Under5_Mortality_Rate_tuned_org.csv' , index = False)

***Skilled_Birth_Attendant_Rate***

In [ ]:
!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
train_data = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
train_data

In [ ]:
training = train_data

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates


In [ ]:
dhsids_training = training_without_duplicates['DHSID']
dhsids_training

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]

In [ ]:
labels = training_without_duplicates.iloc[:, -6:]
labels

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Step 3: Perform KNN imputation to fill NaN values
imputer = SimpleImputer(strategy='mean')  # You can adjust the n_neighbors parameter as needed
training_without_duplicates[labels.columns] = imputer.fit_transform(labels)

# Step 4: Verify the imputed values in 'training' DataFrame
training_without_duplicates

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
sample

In [ ]:
dhsids_sample = sample['DHSID']
dhsids_sample

In [ ]:
training_without_duplicates.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import optuna

# Assuming 'training_imputed' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X = training_without_duplicates.iloc[:, :-6]
y = training_without_duplicates.loc[:, ['Skilled_Birth_Attendant_Rate']]

# Define the number of folds for k-fold cross-validation
n_folds = 5

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1200),  # Expanded range
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
        "random_state": 42,
        "objective": "reg:squarederror",  # Regression task
        "eval_metric": "rmse", # Root Mean Squared Error metric
        "tree_method": "gpu_hist"
        # Add more XGBoost parameters here for tuning (if needed)
        # For example: 'alpha', 'lambda', 'min_split_loss', etc.
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = np.zeros(len(X))
    rmses = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
        oof_predictions[val_idx] = predictions

        rmse = mean_squared_error(y_val, predictions, squared=False)
        rmses.append(rmse)
        
        # Report intermediate result for pruning
        trial.report(rmse, step=len(rmses))
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()


    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = np.mean(rmses)

    return cv_rmse


study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler())
pruner = optuna.pruners.MedianPruner(n_warmup_steps=5)
study_xgb.optimize(objective, n_trials=100)

# After hyperparameter tuning, you can train the final model on the entire training data.
# Use the best hyperparameters obtained from the Optuna study.


In [ ]:
''''best_params =  {'n_estimators': 1148, 
                'learning_rate': 0.009970261274385907,
                 'max_depth': 13, 
                 'subsample': 0.9636818032665447, 
                 'colsample_bytree': 0.6296043325752896, 
                 'min_child_weight': 18,
                 'gamma': 1.970249558007862, 
                 'reg_alpha': 0.5772114039696489,
                 'reg_lambda': 0.18376992370296957}'''

In [ ]:
print('Best hyperparameters:', study_xgb.best_params)
print('Best RMSE:', study_xgb.best_value)

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Skilled_Birth_Attendant_Rate']

In [ ]:
# Get the best hyperparameters for the current label
best_params = study_xgb.best_params
# Train a new model using the best hyperparameters for the current label
final_xgboost_model = xgb.XGBRegressor(**best_params)
final_xgboost_model.fit(X, y)
# Make predictions for the current label
predictions = final_xgboost_model.predict(X_sample)

In [ ]:
cols = ["Skilled_Birth_Attendant_Rate"]

predictions_cat = pd.DataFrame(predictions , columns = cols)
predictions_cat

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)
df

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1)

# Print the merged dataframe
pred_merge_df

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

# Print the merged DataFrame
merged_df

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())
merged_df

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)
merged_df

In [ ]:
merged_df.to_csv('iterSearchsp_900Cols-XGboost-OptunaHyper-Skilled_Birth_Attendant_Rate-CrossVal-GPU.csv' , index = False)

***Stunted_Rate***

In [ ]:
!gdown --id 1d5cqmnbTrbocHmwV4XTQ79aVkls5Kvrn

In [ ]:
!gdown --id 1DweYdpfwMf89p2rS65fvOk5bSvZ6yqrJ

In [ ]:
training = pd.read_csv('/kaggle/working/gee_reduced_training.csv', low_memory = False)
training

In [ ]:
# Assuming you have imported pandas and have your dataframe named 'training'

# Drop duplicate rows based on the 'DHSID' column
training_without_duplicates = training.drop_duplicates(subset='DHSID', keep='first')

# Reset the index
training_without_duplicates = training_without_duplicates.reset_index(drop=True)

# Print the dataframe without duplicate rows and with reset index
training_without_duplicates

In [ ]:
dhsids_training = training_without_duplicates['DHSID']
dhsids_training

In [ ]:
training.columns[training_without_duplicates.isna().any() == True]

In [ ]:
labels = training_without_duplicates.iloc[:, -6:]
labels

In [ ]:
for col in labels.columns:
    print(f'{col}: {labels[col].isna().sum()}')

In [ ]:
training_without_duplicates.dropna(subset=['Stunted_Rate'], inplace = True)

In [ ]:
training_without_duplicates.shape

In [ ]:
sample = pd.read_csv('/kaggle/working/gee_reduced_sample.csv', low_memory = False)
sample

In [ ]:
dhsids_sample = sample['DHSID']
dhsids_sample

In [ ]:
training_without_duplicates.drop('DHSID',axis = 1, inplace = True)
training_without_duplicates

In [ ]:
import pandas as pd
import cupy as cp
import numpy as np
from cuml.ensemble import RandomForestRegressor as cuRandomForestRegressor
from cuml.metrics import mean_squared_error as mean_squared_error_cuml
from sklearn.model_selection import KFold
import optuna

# Assuming 'training_without_duplicates' is your DataFrame with features and labels

# Splitting the data into features (X) and labels (y)
X = training_without_duplicates.iloc[:, :-6]
y = training_without_duplicates.loc[:, 'Stunted_Rate']

# Convert data to cuDF and cuML-compatible format
X_cudf = cp.asarray(X.values.astype(np.float32))
y_cudf = cp.asarray(y.values.astype(np.float32))

# Define the number of folds for k-fold cross-validation
n_folds = 3

# Define the k-fold cross-validator
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "random_state": 42,
        "n_streams": 1
    }

    # Initialize arrays to store the out-of-fold predictions and RMSE scores
    oof_predictions = cp.zeros(len(X_cudf))
    rmses = cp.zeros(n_folds)

    for i, (train_idx, val_idx) in enumerate(kf.split(X_cudf)):
        X_train, X_val = X_cudf[train_idx], X_cudf[val_idx]
        y_train, y_val = y_cudf[train_idx], y_cudf[val_idx]

        model = cuRandomForestRegressor(**params)
        model.fit(X_train, y_train)

        predictions = model.predict(X_val)

        # Store the out-of-fold predictions for potential ensemble techniques (not used in final evaluation)
        oof_predictions[val_idx] = predictions

        rmse = cp.sqrt(mean_squared_error_cuml(y_val, predictions))
        rmses[i] = rmse

        # Prune the trial if the RMSE is too high (early stopping)
        trial.report(float(rmse), step=i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Calculate the overall RMSE for the k-fold cross-validation
    cv_rmse = cp.mean(rmses)

    return cv_rmse

study_rf = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study_rf.optimize(objective, n_trials=250)

# Access the best hyperparameters found by Optuna
best_params = study_rf.best_params

print("Best Hyperparameters:")
print(best_params)

In [ ]:
#Trial 163 finished with value: 19.136056264241535 
'''best_params = {'n_estimators': 997, 
               'max_depth': 20,
               'min_samples_split': 6,
               'min_samples_leaf': 10, 
               'max_features': 0.6031911007542703,
               'random_state': 42
              }'''

In [ ]:
print('Best hyperparameters:', study_rf.best_params)
print('Best RMSE:', study_rf.best_value)

In [ ]:
best_params = study_rf.best_params

In [ ]:
# Train the final Random Forest model on the entire training data using the best hyperparameters
final_rf_model = cuRandomForestRegressor(**best_params)
final_rf_model.fit(X_cudf, y_cudf)

In [ ]:
# Assuming 'sample' is your DataFrame containing the features for prediction
sample2 = sample.drop('DHSID' , axis = 1)
X_sample = sample2.iloc[:, :-6]
y_sample = sample2.loc[:,'Stunted_Rate']

In [ ]:
# Make predictions for the current label
predictions = final_rf_model.predict(X_sample)
predictions

In [ ]:
cols = ["Stunted_Rate"]

predictions_cat = pd.DataFrame(predictions , columns = cols)
predictions_cat

In [ ]:
missing_dhsids = ['EG201403480201',
 'EG201407490204',
 'DHS20180000402',
 'EG201406870404',
 'EG201403980103',
 'EG201404452003',
 'EG201407480104',
 'EG201405500103',
 'DHS20180000554',
 'DHS20180000452',
 'EG201403230104',
 'EG201404100304',
 'EG201404350102',
 'EG201404460201',
 'EG201407240101',
 'EG201406992212',
 'EG201406490204',
 'DHS20180000579',
 'EG201407110104',
 'EG201404950402',
 'EG201406740602',
 'DHS20180000604',
 'EG201405500107',
 'EG201403610201',
 'EG201404340105']

In [ ]:
df = pd.DataFrame(dhsids_sample)
df

In [ ]:
pred_merge_df = pd.concat([df, predictions_cat], axis=1)

# Print the merged dataframe
pred_merge_df

In [ ]:
missing_dhsid_df = pd.DataFrame({'DHSID': missing_dhsids})

# Merge the DataFrames
merged_df = pd.concat([missing_dhsid_df, pred_merge_df], axis=0)

# Print the merged DataFrame
merged_df

In [ ]:
for column in merged_df.columns[1:] :
    merged_df[column] = merged_df[column].fillna(merged_df[column].mean())
merged_df

In [ ]:
merged_df.sort_values('DHSID' , inplace = True)
merged_df

In [ ]:
merged_df.to_csv('RF_Stunted_Rate_tuned_NoNullRowsRemoved.csv' , index = False)